# 🌊 OceanGo: AlphaZero Self-Play Training on Google Colab

This notebook trains the **OceanGo** AlphaZero engine utilizing Google Colab's NVIDIA GPUs (T4, V100, or A100 Tensor Cores).

### Key Benefits:
- **Accelerated GPU Inference**: NVIDIA CUDA Tensor Cores dramatically speed up batched MCTS inference.
- **Direct Google Drive Persistence**: All iteration checkpoints (`policy_value_net_iter_X.pth`) save directly to your Google Drive so nothing is lost upon disconnect.
- **Built-in Dry-Run**: Includes a 10-second verification cell so you never waste compute units on misconfigurations.

## Step 1: Verify Hardware Accelerator
Ensure your Colab runtime is set to **GPU** (`Runtime -> Change runtime type -> Hardware accelerator -> GPU`).

In [ ]:
!nvidia-smi
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device Name: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: GPU accelerator not detected. Please switch to GPU in Runtime -> Change runtime type.")

## Step 2: Mount Google Drive
Mounting Google Drive ensures your checkpoints persist permanently even if Colab disconnects.

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# Create dedicated checkpoint directory on your Google Drive
DRIVE_CHECKPOINT_DIR = '/content/drive/MyDrive/OceanGo/checkpoints'
os.makedirs(DRIVE_CHECKPOINT_DIR, exist_ok=True)
print(f"Checkpoints will be saved directly to: {DRIVE_CHECKPOINT_DIR}")

## Step 3: Clone Repository & Install Dependencies

In [ ]:
# Clone repo (or pull if already present)
import os
if not os.path.isdir('/content/OceanGo'):
    !git clone https://github.com/Ocean48/OceanGo.git /content/OceanGo
else:
    !git -C /content/OceanGo pull

%cd /content/OceanGo
!pip install -r requirements.txt

## Step 4: Dry-Run Verification (Costs ~0 Compute Units)
Runs a quick 10-second end-to-end sanity check to verify CUDA inference, multiprocessing queues, and Google Drive write permissions before starting the full training.

In [ ]:
!python OceanGo.py --dry-run --save-dir /content/drive/MyDrive/OceanGo/checkpoints

## Step 5: Start Full AlphaZero Training
Launches the training pipeline. All models and checkpoints automatically sync to your Google Drive after every iteration, and the runtime **automatically disconnects** when finished so zero Compute Units are wasted on idle time.

### Which Runtime & Preset Should You Choose? (Based on 200 Monthly Compute Units)

| Runtime Setup (`Runtime -> Change runtime type`) | Command Preset | Burn Rate | Time per Loop | 50 Loops Cost & Time | Max Loops per 200 CUs |
| :--- | :--- | :---: | :---: | :---: | :---: |
| 🏆 **T4 GPU + High-RAM ON** *(Best Value — 8 vCPUs)* | `--games 200 --sims 400 --iters 50` | **~2.3 CUs/hr** | **~8–10 mins** | **~18 CUs (~8 hrs)** | **~550+ Loops** |
| **T4 GPU (High-RAM OFF)** *(Longest Clock — 2 vCPUs)* | `--games 200 --sims 400 --iters 50` | **~1.8 CUs/hr** | ~20–25 mins | ~32 CUs (~18 hrs) | ~270 Loops |
| **T4 GPU + High-RAM ON** *(DeepMind Heavy Preset)* | `--games 500 --sims 800 --iters 50` | **~2.3 CUs/hr** | ~35–45 mins | ~75 CUs (~32 hrs) | ~130 Loops |
| **L4 GPU (High-RAM OFF)** *(8 vCPUs, 2x cost)* | `--games 200 --sims 400 --iters 50` | ~4.5 CUs/hr | ~6–8 mins | ~30 CUs (~6.5 hrs) | ~330 Loops |

- **Auto-Resume**: If Colab ever disconnects mid-run, just re-run the notebook—`OceanGo.py` automatically detects your latest `policy_value_net_iter_X.pth` in Google Drive and resumes from `X + 1`.
- **Background Execution**: With Colab Pro / Google AI Pro, you can close this browser tab or shut down your computer while it runs.

In [ ]:
!python OceanGo.py --train \
    --save-dir /content/drive/MyDrive/OceanGo/checkpoints \
    --games 200 \
    --sims 400 \
    --iters 50

# Automatically disconnect and shut down the Colab GPU runtime when training finishes
# so zero Compute Units are wasted on idle time.
from google.colab import runtime
print("Training finished! Automatically disconnecting runtime to save Compute Units...")
runtime.unassign()